# Cluster results — locations and focal mechanisms

A single, cluster-parameterized notebook to view a cluster's **located catalog** and its
**focal mechanisms together**. Set the params below and run top to bottom.

Focal mechanisms require a **phasenet_plus** run (the PhaseNet+ picker emits first-motion polarity
+ S/P amplitude that SKHASH needs); point `RUN_SUFFIX` at that run's output tree. Locations alone
work for any picker.

In [ ]:
%matplotlib inline
import os, sys
# Assumes the notebook runs from pipeline/notebooks/ ; otherwise set PYTHONPATH=<repo root>.
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from pipeline import config, viz

# ------------------------------- PARAMS (edit & re-run) -------------------------------
CLUSTER    = "gwangyang"        # gwangyang | kimcheon | jangsung | gyeongju
RUN_SUFFIX = "_pnplus"          # output tree = runs/<cluster><suffix>; "" = the default run
VELMODEL   = "kim1983"          # velocity model whose .sum / mechanisms to show

cfg0 = config.load_cluster(CLUSTER)
cfg  = config.tune(cfg0, output_root=os.path.join(config.RUNS_ROOT, f"{CLUSTER}{RUN_SUFFIX}")) \
       if RUN_SUFFIX else cfg0
print(f"{cfg.region}: outputs -> {cfg.output_root}")

## 1. Locations

In [ ]:
viz.map_catalog(cfg, velmodel=VELMODEL, source="sum"); plt.show()
viz.depth_sections(cfg, velmodel=VELMODEL, source="sum"); plt.show()
viz.cumulative_events(cfg, velmodel=VELMODEL); plt.show()

In [ ]:
# relocated catalog, if this run has HypoDD output (dt.ct / dt.cc)
if os.path.exists(os.path.join(config.dtct_dir(cfg), "hypoDD.reloc")):
    viz.map_catalog(cfg, velmodel=VELMODEL, source="reloc"); plt.show()
else:
    print("No HypoDD reloc for this run — showing absolute (.sum) locations only.")

## 2. Focal mechanisms

The table is one row per event (best quality kept). `map_mechanisms` shows the **locations and
focal mechanisms together**: located epicenters as depth-coloured dots, with the high-confidence
(quality A/B) beachballs offset on a ring around the cluster (leader line to each true epicenter)
so a tight cluster stays legible.

In [ ]:
tbl = viz.mechanism_table(cfg, VELMODEL)
display(tbl)
viz.map_mechanisms(cfg, VELMODEL); plt.show()

In [ ]:
# per-event beachball gallery (SKHASH plots) for the high-confidence events
from matplotlib import image as mpimg
mech = pd.read_csv(config.fm_mech_csv(cfg, VELMODEL)).drop_duplicates("event_id")
e2c  = dict(zip(mech.event_id.astype(str), mech.cuspid.astype(int)))
hi   = tbl[tbl.quality.isin(cfg.fm_quality_keep)] if len(tbl) else tbl
ids  = list(hi.event_id.astype(str))
out  = config.fm_out_dir(cfg, VELMODEL)
if ids:
    ncol = min(3, len(ids)); nrow = (len(ids) + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 3.8 * nrow), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, eid in zip(axes.ravel(), ids):
        r = hi[hi.event_id.astype(str) == eid].iloc[0]
        png = os.path.join(out, f"{e2c.get(eid)}.png")
        if os.path.exists(png):
            ax.imshow(mpimg.imread(png))
        ax.set_title(f"{eid}   {r.quality}\ns/d/r = {r.strike:.0f}/{r.dip:.0f}/{r.rake:.0f}",
                     fontsize=9)
    fig.suptitle(f"{cfg.region} — high-confidence focal mechanisms (SKHASH beachballs)", fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print("No high-confidence (A/B) mechanisms for this run "
          "(needs a phasenet_plus focal_mechanism run).")

## 3. Interpretation

- **Quality A/B** = well-constrained ("fairly high confidence"); C/D are under-constrained and shown
  for context only. SKHASH grades on polarity misfit, station-distribution ratio, azimuthal/takeoff
  gaps, and mechanism probability.
- **Polarity** (vertical first motion) is the robust signal; the vertical-component **S/P ratio** is a
  secondary enhancement (`cfg.fm_use_sp_ratio`). Re-run with `fm_use_sp_ratio=False` for a
  polarity-only comparison.
- Consistent beachballs across events indicate a coherent source process on a common fault geometry.
- To regenerate: run `picking` (`--picker phasenet_plus`), `hypoinverse`, then the `focal_mechanism`
  stage for this cluster (see the top-level README).